# Data Processing · Apache Spark, DataFrames API II
**IE University | 80 minutes**

| Minutes | Activity |
|---|---|
| 0–10 | Live discussion · Are you following me? |
| 10–40 | Lecture · More on aggregations in Spark |
| 40–70 | Lecture · Joins and UDFs |
| 70–80 | Challenge · Did you get it? |

By the end, you should be able to build grouped metrics and subtotals, predict how joins change row counts, and compare a Python UDF with an equivalent Spark SQL expression.

**Before class:** Open this notebook in Jupyter with a Python 3.10–3.12 kernel and Java 17 or 21. If needed, run `%pip install pyspark==4.0.1` in a separate cell and restart the kernel. The synthetic livestock dataset is embedded. Each weighing row represents an event, so an animal can appear more than once.

In [12]:
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import StringType

spark = (
    SparkSession.builder.master("local[2]")
    .appName("IE University - DataFrames API II")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark:", spark.version)

farms = spark.createDataFrame(
    [("F1", "Gredos"), ("F2", "Encinar"), ("F3", "Valle")],
    "farm_id string, farm_name string",
)
weighings = spark.createDataFrame(
    [
        (1, "A1", "F1", "heifer", 280),
        (2, "A1", "F1", "heifer", 320),
        (3, "A2", "F1", "cow", 510),
        (4, "A3", "F1", "cow", None),
        (5, "A4", "F2", "heifer", 300),
        (6, "A5", "F2", "cow", 470),
        (7, "A6", "F9", "cow", 430),
        (8, "A7", None, "heifer", 310),
        (9, "A8", "F2", None, 260),
    ],
    "event_id int, animal_id string, farm_id string, category string, weight_kg int",
)
farms.orderBy("farm_id").show()
weighings.orderBy("event_id").show()

Spark: 4.0.5
+-------+---------+
|farm_id|farm_name|
+-------+---------+
|     F1|   Gredos|
|     F2|  Encinar|
|     F3|    Valle|
+-------+---------+

+--------+---------+-------+--------+---------+
|event_id|animal_id|farm_id|category|weight_kg|
+--------+---------+-------+--------+---------+
|       1|       A1|     F1|  heifer|      280|
|       2|       A1|     F1|  heifer|      320|
|       3|       A2|     F1|     cow|      510|
|       4|       A3|     F1|     cow|     NULL|
|       5|       A4|     F2|  heifer|      300|
|       6|       A5|     F2|     cow|      470|
|       7|       A6|     F9|     cow|      430|
|       8|       A7|   NULL|  heifer|      310|
|       9|       A8|     F2|    NULL|      260|
+--------+---------+-------+--------+---------+



## 0–10 min · Are you following me?

**0–3:** What does one row represent? Which identifier can repeat?  
**3–7:** Predict how many events have a recorded weight of at least 300 kg. What happens to a missing measurement? Which line triggers execution?  
**7–10:** Run the cell and compare predictions. The filter keeps rows where the comparison is true. A null weight makes the comparison unknown.

In [13]:
heavy_events = weighings.filter(F.col("weight_kg") >= 300)  # Transformation
heavy_events.orderBy("event_id").show()                     # Action: six rows

+--------+---------+-------+--------+---------+
|event_id|animal_id|farm_id|category|weight_kg|
+--------+---------+-------+--------+---------+
|       2|       A1|     F1|  heifer|      320|
|       3|       A2|     F1|     cow|      510|
|       5|       A4|     F2|  heifer|      300|
|       6|       A5|     F2|     cow|      470|
|       7|       A6|     F9|     cow|      430|
|       8|       A7|   NULL|  heifer|      310|
+--------+---------+-------+--------+---------+



## 10–40 min · More on aggregations in Spark

### 10–20 · Counts, grain, and missing values

Predict the three counts for F1. The three expressions count rows, non-null weights, and distinct non-null animal IDs, respectively. The mean skips null weights. Repeated weighings give an animal more influence on an event-level mean.

In [14]:
metrics = weighings.groupBy("farm_id").agg(
    F.count("*").alias("events"),
    F.count("weight_kg").alias("recorded_weights"),
    F.countDistinct("animal_id").alias("animals"),
    F.round(F.avg("weight_kg"), 2).alias("mean_kg"),
)
metrics.orderBy("farm_id").show()

+-------+------+----------------+-------+-------+
|farm_id|events|recorded_weights|animals|mean_kg|
+-------+------+----------------+-------+-------+
|   NULL|     1|               1|      1|  310.0|
|     F1|     4|               3|      3|  370.0|
|     F2|     3|               3|      3| 343.33|
|     F9|     1|               1|      1|  430.0|
+-------+------+----------------+-------+-------+



F1 has four events, three recorded weights and three distinct animals. To give each measured animal equal weight, calculate its average first, then average those results by farm.

In [15]:
animal_means = weighings.groupBy("farm_id", "animal_id").agg(
    F.avg("weight_kg").alias("animal_mean_kg")
)
animal_means.groupBy("farm_id").agg(
    F.round(F.avg("animal_mean_kg"), 2).alias("mean_of_animal_means_kg")
).orderBy("farm_id").show()

+-------+-----------------------+
|farm_id|mean_of_animal_means_kg|
+-------+-----------------------+
|   NULL|                  310.0|
|     F1|                  405.0|
|     F2|                 343.33|
|     F9|                  430.0|
+-------+-----------------------+



### 20–30 · Conditional metrics

What share of **recorded weights** reaches 300 kg? Decide whether missing weights belong in the denominator. Here the denominator includes recorded weights only. A group with zero recorded weights would have an undefined share, which `try_divide` represents as null.

In [16]:
conditional = weighings.groupBy("farm_id").agg(
    F.count("weight_kg").alias("recorded_weights"),
    F.count_if(F.col("weight_kg") >= 300).alias("at_least_300"),
    F.count_if(F.col("weight_kg").isNull()).alias("missing_weights"),
)
conditional.withColumn(
    "pct_at_least_300",
    F.round(100 * F.try_divide("at_least_300", "recorded_weights"), 1),
).orderBy("farm_id").show()

+-------+----------------+------------+---------------+----------------+
|farm_id|recorded_weights|at_least_300|missing_weights|pct_at_least_300|
+-------+----------------+------------+---------------+----------------+
|   NULL|               1|           1|              0|           100.0|
|     F1|               3|           2|              1|            66.7|
|     F2|               3|           2|              0|            66.7|
|     F9|               1|           1|              0|           100.0|
+-------+----------------+------------+---------------+----------------+



### 30–40 · Subtotals with rollup

Read the grouping ID to distinguish data nulls from subtotal markers. The ordered dimensions produce detail (0), farm subtotals (1), and a grand total (3). Explain the two F2 rows with null category. Which row counts every input event?

In [ ]:
weighings.rollup("farm_id", "category").agg(
    F.count("*").alias("events"),
    F.grouping_id("farm_id", "category").alias("grouping_level"),
).orderBy("grouping_level", "farm_id", "category").show()

## 40–70 min · Joins and UDFs

### 40–52 · Choose the join direction and type

| Join | Question with weighings on the left |
|---|---|
| Inner | Which events match a registered farm? |
| Left | Can I keep every event and attach farm details where available? |
| Left semi | Which events have a match? Return only event columns. |
| Left anti | Which events lack a match? Return only event columns. |

Predict the row counts. Ordinary equality never matches a null key. Reverse the anti join to find registered farms without events.

In [ ]:
for how in ("inner", "left", "left_semi", "left_anti"):
    result = weighings.join(farms, "farm_id", how)
    print(f"{how}: {result.count()} rows")

weighings.join(farms, "farm_id", "left").select(
    "event_id", "farm_id", "farm_name"
).orderBy("event_id").show()
weighings.join(farms, "farm_id", "left_anti").orderBy("event_id").show()
farms.join(weighings, "farm_id", "left_anti").show()

### 52–58 · Duplicate keys and row multiplication

Suppose the farm registry accidentally contains two F1 rows. Predict the effect on the join and a downstream sum. Check uniqueness on the full join key. If multiple registry rows are valid, define a selection rule, such as matching on the effective date as well.

In [ ]:
duplicate_registry = farms.unionByName(
    spark.createDataFrame([("F1", "Gredos (old name)")], farms.schema)
)
duplicate_registry.groupBy("farm_id").count().filter("count > 1").show()
print("Clean left join:", weighings.join(farms, "farm_id", "left").count())
print("Duplicate key:", weighings.join(duplicate_registry, "farm_id", "left").count())

### 58–70 · Write a Python UDF and compare plans

A scalar Python UDF returns one value per row. Declare the output type and decide how null inputs should behave. Here 300 kg is an arbitrary reporting threshold.

Find an event below 300, one exactly at 300, and one with a missing weight. Compare the two plans. The regular Python UDF introduces a Python evaluation step (typically `BatchEvalPython`). The equivalent built-in expression evaluates inside Spark SQL and avoids that Python boundary. Use the plans to explain the difference. This dataset is too small for meaningful timing comparisons.

In [ ]:
@F.udf(returnType=StringType(), useArrow=False)
def weight_band(weight_kg):
    if weight_kg is None:
        return "missing"
    return "under 300" if weight_kg < 300 else "300 or more"


python_bands = weighings.select("event_id", weight_band("weight_kg").alias("python_band"))
native_bands = weighings.select(
    "event_id",
    F.when(F.col("weight_kg").isNull(), "missing")
    .when(F.col("weight_kg") < 300, "under 300")
    .otherwise("300 or more")
    .alias("native_band"),
)
comparison = python_bands.join(native_bands, "event_id")
comparison.orderBy("event_id").show()
assert comparison.filter(~F.col("python_band").eqNullSafe(F.col("native_band"))).count() == 0
python_bands.explain()
native_bands.explain()

## 70–80 min · Did you get it?

**70–72:** Choose a join and predict how many rows the final report should contain.  
**72–77:** Build one row per **registered farm**, including farms with zero events. Return `farm_id`, `farm_name`, `event_count`, `recorded_weights`, and `at_least_300`. Use a built-in expression for the threshold count.  
**77–80:** Run the checks, then explain why counting every joined row would give Valle an event and where the two unmatched events went.

<details>
<summary>Hint</summary>

Put the farm registry on the left. Count a column that identifies an event and becomes null when the join finds no match.

</details>

In [ ]:
# Replace None with your DataFrame expression.
report = None

In [ ]:
def check_report(result):
    expected = [
        ("F1", "Gredos", 4, 3, 2),
        ("F2", "Encinar", 3, 3, 2),
        ("F3", "Valle", 0, 0, 0),
    ]
    ordered = result.select(
        "farm_id", "farm_name", "event_count", "recorded_weights", "at_least_300"
    ).orderBy("farm_id")
    ordered.show()
    assert [tuple(row) for row in ordered.collect()] == expected, "Check the join and counts."
    print("All three farm rows match.")


if report is None:
    print("Build report in the preceding cell, then rerun this check.")
else:
    check_report(report)

<details>
<summary>Worked solution · open after attempting the challenge</summary>

~~~python
solution = (
    farms.join(weighings, "farm_id", "left")
    .groupBy("farm_id", "farm_name")
    .agg(
        F.count("event_id").alias("event_count"),
        F.count("weight_kg").alias("recorded_weights"),
        F.count_if(F.col("weight_kg") >= 300).alias("at_least_300"),
    )
)
check_report(solution)
~~~

Valle contributes one joined row with null event fields, so counting event IDs gives zero. The events with an unknown or missing farm ID are excluded from the farm report. The event-to-registry anti join can surface both for follow-up.

</details>

### Learning resources

- Use this notebook alongside your session’s **DataFrames API II slides**.
- [PySpark installation](https://spark.apache.org/docs/4.0.1/api/python/getting_started/install.html)
- [DataFrame API reference](https://spark.apache.org/docs/4.0.1/api/python/reference/pyspark.sql/dataframe.html)
- [PySpark SQL functions](https://spark.apache.org/docs/4.0.1/api/python/reference/pyspark.sql/functions.html)

Run `spark.stop()` when you finish to release the local Spark session.